# Taxi Trip Duration Pipeline

This notebook now acts as an orchestrator for the modularized codebase in the  directory.

In [1]:
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
# Add src to the path so we can import our modules
sys.path.append(os.path.abspath('src'))

from data.data_loader import load_data, get_target
from features.build_features import preprocess_features
from models.train_model import train_xgboost, train_linear_model, evaluate_with_cv
from models.predict_model import load_model, predict

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data

## 2. Exploratory Data Analysis & Outlier Detection

In [ ]:
# Check target distribution
plt.figure(figsize=(10, 6))
sns.histplot(np.log1p(df_train['trip_duration']), bins=100, kde=True)
plt.title('Distribution of Log(Trip Duration)')
plt.xlabel('Log(Duration + 1)')
plt.show()

# NYC Boundaries
# Longitude: -74.05 to -73.75
# Latitude: 40.63 to 40.85
def filter_outliers(df):
    initial_shape = df.shape[0]
    df = df[(df['trip_duration'] > 60) & (df['trip_duration'] < 3600*12)] # 1 min to 12 hours
    df = df[(df['pickup_longitude'] > -74.05) & (df['pickup_longitude'] < -73.75)]
    df = df[(df['pickup_latitude'] > 40.63) & (df['pickup_latitude'] < 40.85)]
    df = df[(df['dropoff_longitude'] > -74.05) & (df['dropoff_longitude'] < -73.75)]
    df = df[(df['dropoff_latitude'] > 40.63) & (df['dropoff_latitude'] < 40.85)]
    print(f"Removed {initial_shape - df.shape[0]} outliers.")
    return df

df_train = filter_outliers(df_train)
df_val = filter_outliers(df_val)

In [2]:
data_dir = 'data_set'
df_train, df_val, df_test = load_data(data_dir)
print(f'Train shape: {df_train.shape}')
print(f'Validation shape: {df_val.shape}')
print(f'Test shape: {df_test.shape}')


Train shape: (1000000, 10)
Validation shape: (229319, 10)
Test shape: (229322, 10)


## 2. Feature Engineering
We will extract target variables and compute spatial and temporal features.

In [3]:
X_train, y_train = get_target(df_train)
X_val, y_val = get_target(df_val)
X_test, _ = get_target(df_test)

X_train = preprocess_features(X_train)
X_val = preprocess_features(X_val)
X_test = preprocess_features(X_test)

print(f'Engineered Train shape: {X_train.shape}')


Engineered Train shape: (1000000, 14)


## 3. Model Training (XGBoost)

In [4]:
# This will train the model and save it to the models/ directory
model, val_rmse = train_xgboost(X_train, y_train, X_val, y_val)


Training XGBoost Model...


[0]	validation_0-rmse:0.74254	validation_1-rmse:0.74792


[10]	validation_0-rmse:0.51061	validation_1-rmse:0.51699


[20]	validation_0-rmse:0.43650	validation_1-rmse:0.44357


[30]	validation_0-rmse:0.41574	validation_1-rmse:0.42375


...

[170]	validation_0-rmse:0.36984	validation_1-rmse:0.39601


[180]	validation_0-rmse:0.36844	validation_1-rmse:0.39552


[190]	validation_0-rmse:0.36692	validation_1-rmse:0.39514


[199]	validation_0-rmse:0.36559	validation_1-rmse:0.39488


Validation RMSE (log scale): 0.3949
Model saved to models/xgb_model.pkl


## 4. Model Comparison (Ridge, Lasso, CV)

In [ ]:
# Train Ridge
model_ridge, rmse_ridge = train_linear_model(X_train, y_train, X_val, y_val, model_type='ridge')

# Train Lasso
model_lasso, rmse_lasso = train_linear_model(X_train, y_train, X_val, y_val, model_type='lasso')

# Cross-Validation comparison (using a sample for speed if necessary)
# cv_results_xgb = evaluate_with_cv(X_train.iloc[:50000], y_train.iloc[:50000], model_type='xgb')
# cv_results_ridge = evaluate_with_cv(X_train.iloc[:50000], y_train.iloc[:50000], model_type='ridge')


## 4. Prediction

In [5]:
# Generate predictions on the test set
test_predictions = predict(model, X_test)
print(test_predictions[:5])


Generating predictions...


[ 404.03558  523.92737  883.1215  2235.6963  1612.469  ]
